In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import datetime
import ipynbname

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables'
IN_FILE = 'pubmed_results_Jamalah_730.xlsx'
OUT_FILE = 'pubmed_results_Jamalah_730_with_gpt.xlsx'
EXTRA_SOURCES = {
    '30': 'final30.xlsx',
    '495': '495.xlsx',
}
LOG_FILE = 'log.txt'

In [ ]:
CRITERIA = '''
## INCLUSION CRITERIA
Studies that focus on barriers, enablers, or facilitators of MMR or HPV vaccination
Geography- global, Conducted in developed countries (mid-to-high income countries)
Study Design: Randomized controlled trials (RCTs), quasi-experimental, ecological, or observational cohort studies•	Study Design: Randomized controlled trials (RCTs), quasi-experimental, ecological, or observational cohort studies
Peer reviewed articles
Articles published in English
Articles published since 2010
Population: children, adolescents, parents, health care workers in mid-to-high income countries
Excluding: interventions, studies from U.S. (case by case- depending on population looked at), studies with health systems very different
'''


In [ ]:
df = pd.read_excel(os.path.join(DIR, IN_FILE))

In [ ]:
expected_columns = ['title', 'abstract', 'authors']
for column in expected_columns:
    assert column in df.columns, f'Column {column} is missing.'
assert 'year' in df.columns or 'publication_date' in df.columns, 'Must have either year or publication_date'

In [ ]:
columns_to_keep = expected_columns[:]
for column in ['year', 'publication_date']:
    if column in df.columns:
        columns_to_keep.append(column)
df = df[columns_to_keep]
if 'year' not in df.columns:
    df['year'] = pd.to_datetime(df.publication_date).dt.year

In [ ]:
title_extracts = df.title.apply(extract_letters)
is_in_cols = []
for name, filename in EXTRA_SOURCES.items():
    source_df = pd.read_excel(os.path.join(DIR, filename))
    if 'title' in source_df.columns:
        source_titles = source_df.title
    elif 'Title' in source_df.columns:
        source_titles = source_df.Title
    else:
        raise ValueError(f'File {filename} has no title column')
    source_title_extracts = source_titles.apply(extract_letters)

    new_col = f'is_in_{name}'
    df[new_col] = title_extracts.isin(source_title_extracts)
    is_in_cols.append(new_col)
df.head()

In [ ]:
df[is_in_cols].sum()

In [ ]:
eligibility_abstract = gpt.check_eligibility(df, criteria=CRITERIA)

In [ ]:
eligibility_abstract.rename(columns={'reasoning': 'reasoning_title_abstract', 'decision': 'decision_title_abstract'}, inplace=True)
eligibility_abstract.columns

In [ ]:
no_abstract = eligibility_abstract.copy()
no_abstract.loc['abstract', :] = np.nan
no_abstract_res = gpt.check_eligibility(no_abstract, criteria=CRITERIA, batch_size=100)

In [ ]:
part = no_abstract_res[['reasoning', 'decision']].copy()
part.rename(columns={'reasoning': 'reasoning_title', 'decision': 'decision_title'}, inplace=True)
eligibility_all = pd.concat([eligibility_abstract, part], axis=1)
eligibility_all.head()

In [ ]:
eligibility_all.to_excel(os.path.join(DIR, OUT_PATH), index=False)

## Confusion matrix

In [ ]:
#eligibility_all = pd.read_excel(OUT_PATH)
eligibility_all.columns

In [ ]:
decision = eligibility_all.decision_title
gt = eligibility_all.in_495 == 1

In [ ]:
def confusion_matrix(decision, gt):
    df = pd.DataFrame({'decision': decision, 'truth': gt})
    df['val'] = 1
    res = df.groupby(['decision', 'truth']).sum().reset_index(drop=False)
    return res.pivot(index='decision', columns='truth', values='val')    

In [ ]:
log_file = open(os.path.join(DIR,   LOG_NAME), 'at')

In [ ]:
log_file.write(f'\n==================== {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")} ====================\n\n')
log_file.write(f'{ipynbname.path()}\n')
log_file.write(f'Input: {IN_PATH}\n')
log_file.write(f'Output: {OUT_PATH}\n')

In [ ]:
log_file.write(f'\nSystem prompt:\n\n{gpt.SYSTEM_PROMPT}\n\n')
log_file.write(f'\nCriteria:\n\n{gpt.JUMALAH_CRITERIA}\n\n')

Title

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title, gt=(eligibility_all.in_495 == 1))
confusion

In [ ]:
log_file.write(f'\nConfusion for titles:\n{confusion}\n\n')

Title and abstract

In [ ]:
# confusion = confusion_matrix(decision=eligibility_all['decision_title_abstract'], gt=eligibility_all.in_86)
# confusion

In [ ]:
# log_file.write(f'\nConfusion for titles:\n{confusion}\n\n')

Title, abstract, AND

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title_abstract & eligibility_all.decision_title, gt=eligibility_all.in_final_30)
confusion

In [ ]:
log_file.write(f'\nConfusion for AND:\n{confusion}\n\n')

In [ ]:
log_file.close()